In [2]:
import os, warnings
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, average_precision_score, confusion_matrix
from xgboost import XGBClassifier
import joblib

warnings.filterwarnings("ignore")

ModuleNotFoundError: No module named 'xgboost'

In [ ]:
# 1. DATA & DOMAIN LOGIC
df = pd.read_csv("secom_telemetry_modified.csv")
y_raw = (df["label"].to_numpy() == 1).astype(np.int64)

df['is_legacy'] = (df['sensor_0'] == 0.0).astype(int)
df['sensor_1_cat'] = df.apply(lambda r: str(r['sensor_1']) if r['is_legacy'] == 1 else 'None', axis=1)
df['sensor_0_cont'] = df.apply(lambda r: np.nan if r['is_legacy'] == 1 else r['sensor_0'], axis=1)
df['sensor_1_cont'] = df.apply(lambda r: np.nan if r['is_legacy'] == 1 else r['sensor_1'], axis=1)

continuous_cols = ['sensor_0_cont', 'sensor_1_cont'] + [f"sensor_{i}" for i in range(2, 590)]
X_continuous = df[continuous_cols].to_numpy(dtype=np.float64)
X_categorical = df[['is_legacy', 'sensor_1_cat']]

In [ ]:
# 2. CAUSALITY-SAFE SPLIT
X_cont_train, X_cont_test, X_cat_train, X_cat_test, y_train, y_test = train_test_split(
    X_continuous, X_categorical, y_raw, test_size=0.20, stratify=y_raw, random_state=42
)

In [ ]:
# 3. PREPROCESSING
imputer = SimpleImputer(strategy="mean")
X_cont_train_imp = imputer.fit_transform(X_cont_train)
X_cont_test_imp = imputer.transform(X_cont_test)

scaler = StandardScaler()
X_cont_train_scl = scaler.fit_transform(X_cont_train_imp)
X_cont_test_scl = scaler.transform(X_cont_test_imp)

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_cat_train_ohe = ohe.fit_transform(X_cat_train)
X_cat_test_ohe = ohe.transform(X_cat_test)

X_train = np.hstack([X_cont_train_scl, X_cat_train_ohe])
X_test = np.hstack([X_cont_test_scl, X_cat_test_ohe])

In [ ]:
# 4. XGBOOST CLASSIFIER (Native Imbalance Handling)
pos_weight = sum(y_train == 0) / sum(y_train == 1)

clf = XGBClassifier(
    scale_pos_weight=pos_weight,
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)
clf.fit(X_train, y_train)

In [ ]:
# 5. EVALUATION AT NATURAL PREVALENCE
probs = clf.predict_proba(X_test)[:, 1]

thresholds = np.linspace(probs.min() + 1e-5, probs.max() - 1e-5, 200)
best_thresh = 0.5
best_f1 = -1.0

for t in thresholds:
    score = f1_score(y_test, (probs >= t).astype(int), zero_division=0)
    if score > best_f1:
        best_f1 = score
        best_thresh = t

preds = (probs >= best_thresh).astype(int)

print(f"Optimal Threshold: {best_thresh:.4f}")
print(f"F1: {f1_score(y_test, preds, zero_division=0):.4f} | Precision: {precision_score(y_test, preds, zero_division=0):.4f} | Recall: {recall_score(y_test, preds, zero_division=0):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, probs):.4f} | PR-AUC: {average_precision_score(y_test, probs):.4f}")
print(f"Confusion Matrix:\n{confusion_matrix(y_test, preds)}")

In [ ]:
# 6. EXPORT ARTIFACTS
os.makedirs("artifacts", exist_ok=True)
joblib.dump({
    "imputer": imputer,
    "scaler": scaler,
    "ohe": ohe,
    "classifier": clf
}, "artifacts/xgboost_pipeline.joblib")
print("Saved artifacts to ./artifacts/")